# 第05课 - 主动检索增强生成模型 (Agentic RAG)


## 设置

本笔记本演示了使用 Microsoft Agent Framework 的 Agentic RAG（检索增强生成）模式。

**先决条件：**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — 你的 Azure AI 搜索服务端点
- `AZURE_SEARCH_API_KEY` — 你的 Azure AI 搜索 API 密钥
- 通过环境变量配置的 Azure OpenAI 部署
- 已登录 Azure CLI（`az login`）


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 什么是 Agentic RAG？

传统的 RAG 遵循固定流程：检索文档，然后生成回复。**Agentic RAG** 更进一步，赋予代理在<strong>何时</strong>以及<strong>如何</strong>检索信息上的自主权。

通过 Agentic RAG，代理可以：
- <strong>决定</strong>在回答问题前是否需要检索
- <strong>选择</strong>查询的数据源或工具
- <strong>评估</strong>检索结果，如果首次尝试不足则进行后续检索
- <strong>结合</strong>多次检索步骤的信息，形成连贯的回答

这使得代理比静态的先检索后生成流程更加灵活和精准。


## 创建搜索工具

在 Agentic RAG 中，外部数据源被封装为代理可以按需调用的<strong>工具</strong>。这使得代理将检索视为其可以执行的另一种操作，而不是强制步骤。

下面我们定义了一个旅游知识库，并将其作为代理可以调用的工具，以查找目的地信息。


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## 构建 RAG 代理

现在我们创建一个被指示为<strong>始终在回答前检索信息</strong>的代理。该代理使用 `search_travel_knowledge` 工具，将其回答建立在知识库的基础上，而不是依赖其自身的训练数据。


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## 迭代检索 — 制作者-审核者模式

Agentic RAG 的一个关键优势是<strong>迭代检索</strong>。代理可以执行多轮搜索来验证、完善或扩展其初步发现——类似于“制作者-审核者”的工作流程：

1. <strong>制作者步骤</strong>：代理检索初始信息并草拟回应。
2. <strong>审核者步骤</strong>：代理执行额外的检索以核实细节或填补空白。

下面，代理被问及一个需要比较多个目的地的问题，促使它进行多次搜索。


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## 总结

在本课中，您学习了如何使用 Microsoft Agent Framework 构建一个<strong>自主检索生成（Agentic RAG）</strong>系统：

- <strong>自主检索生成（Agentic RAG）</strong>允许代理自主决定何时检索信息，使检索变得动态而非固定。
- <strong>工具作为数据源</strong>：将外部知识库（如 Azure AI Search）封装为代理可以调用的工具。
- <strong>迭代检索</strong>：制造者-审核者模式使代理能够执行多轮检索 —— 搜索、验证和完善 —— 然后生成最终答案。

在生产环境中，您会用真实的 Azure AI Search 索引替换内存中的 `TRAVEL_KNOWLEDGE_BASE`，以处理大规模的旅游文档检索。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
